In [2]:
!pip list | grep gradio

# 環境預設有 Gradio

gradio                                   6.5.1
gradio_client                            2.0.3


In [1]:
# 強制更新庫以支援 router 協議與 post 方法
!pip install --upgrade huggingface_hub gradio requests

# 跑得起來的 Code - 使用 Hugging Face API

## 移除 "think" 標籤

In [17]:
import gradio as gr
from huggingface_hub import InferenceClient
from google.colab import userdata
import re

HF_TOKEN = userdata.get("HF_TOKEN").strip()

# 關鍵：model 要加上 :hf-inference
MODEL = "HuggingFaceTB/SmolLM3-3B:hf-inference"

client = InferenceClient(api_key=HF_TOKEN)  # 讓它走 router / providers

# 解決在 response 中出現 <think> 的問題

def get_final_answer(text: str) -> str:
    if not text:
        return ""

    # 最穩：結論在 </think> 後
    m = re.search(r"</think>\s*(.*)$", text, flags=re.DOTALL | re.IGNORECASE)
    if m:
        ans = m.group(1).strip()
        if ans:
            return ans

    # 有 <think> 但沒有 </think>：通常是 max_tokens 太小被截斷
    if re.search(r"<think>", text, flags=re.IGNORECASE):
        return "（回答尚未生成完成：輸出被截斷。請提高 max_tokens，或改用要求 <final> 的輸出格式。）"

    # 沒有 think 標籤：原樣回傳
    return text.strip()

def generate_text(prompt: str):
    try:
        resp = client.chat_completion(
            model=MODEL,
            messages=[
                # 這句是「明確要求只輸出答案」
                {"role": "system", "content": "請直接輸出最終答案，不要輸出<think>或任何推理過程。"},
                {"role": "user", "content": prompt},
            ],
            max_tokens=1000,
            temperature=0.7,
        )

        content = resp.choices[0].message["content"]
        return get_final_answer(content)
    except Exception as e:
        return f"錯誤詳情: {e}"

with gr.Blocks() as demo:
    # 介面大標題
    gr.Markdown("### Hugging Face (Chat Completion via Router)")
    # 輸入框與輸出框
    in_box = gr.Textbox(label="輸入 Prompt", value="The best programming language is")
    out_box = gr.Textbox(label="生成結果")
    # 執行送出按鈕
    gr.Button("生成").click(fn=generate_text, inputs=in_box, outputs=out_box)

demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ffc0058b9f2db2ef71.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ffc0058b9f2db2ef71.gradio.live


## 未移除 "think" 標籤區塊的完整結果

```
<think>
Okay, so the user is asking, "The best programming language is..." Hmm, I need to figure out how to answer this. First, I should consider what makes a programming language "the best." It's subjective, so there's no one definitive answer. But maybe I can explain the different factors people consider when choosing a programming language.

Let me start by thinking about the main reasons people pick a language. Some might care about ease of use and rapid prototyping, others about performance, or maybe the community and job opportunities. Different languages excel in different areas. For example, Python is great for beginners and data science, JavaScript for web development, C++ for performance-critical applications, and so on.

Wait, the user might be expecting a specific answer, but since it's subjective, I should mention that. Also, maybe provide some examples of languages and their strengths. But I need to make sure I don't favor one over others. Let me check if there's any recent trends or surveys that might influence this. For instance, Stack Overflow's annual survey or TIOBE index could be referenced, but they might not be the most accurate for all users.

Another angle: the user might be looking for a recommendation based on their specific needs. But since the question is general, I should keep it broad. I should also mention that the "best" language depends on the user's goals, such as personal interest, industry, or the type of project they're working on. Maybe list a few languages with their strengths and suggest that the best one is the one that fits their needs best.

I should avoid making a definitive statement and instead present the information in a way that's helpful. Let me structure the answer by listing popular languages, their use cases, and the factors that influence their popularity. Then conclude by emphasizing that there's no single best language but that the choice depends on the user's context.
</think>

The best programming language is subjective and depends on individual needs, such as the project's purpose, performance requirements, ease of learning, and the developer's goals. Here are key considerations for popular languages:

1. **Python**: Ideal for beginners, data science, machine learning, and rapid prototyping.
2. **JavaScript**: Essential for web development (front-end and back-end) and web applications.
3. **Java**: Widely used for enterprise applications, Android development, and large-scale systems.
4. **C++**: Known for high-performance applications, game development, and systems programming.
5. **C#**: Popular in game development (Unity) and enterprise software.
6. **Go (Golang)**: Efficient for concurrent programming and cloud computing.
7. **Rust**: Focuses on memory safety and performance, with growing adoption in systems programming.
8. **Swift**: Designed for iOS and macOS development, with strong community support.
9. **TypeScript**: TypeScript is a superset of JavaScript, used for large-scale JavaScript projects.

Choose the language that aligns with your project's requirements, such as performance, ecosystem, or industry standards. There is no single "best" language—it depends on context.
```